In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

## 1. `regexp_extract()`

Extracts a specific group matched by a regex pattern from a string column.

**Syntax:** `regexp_extract(str, pattern, idx)`
- `str`: Column to extract from
- `pattern`: Regular expression pattern
- `idx`: Group index to extract (0 = entire match, 1+ = capture groups)

In [54]:
from pyspark.sql.functions import regexp_extract

data = [("john.doe@example.com",), ("jane.smith@test.org",)]
df = spark.createDataFrame(data, ["email"])

# Extract username from email
df_extracted = df.withColumn("username", regexp_extract("email", r"([^@]+)@", 1))
df_extracted.show()

+--------------------+----------+
|               email|  username|
+--------------------+----------+
|john.doe@example.com|  john.doe|
| jane.smith@test.org|jane.smith|
+--------------------+----------+



In [57]:
# Example with Multiple Capture Groups

data = [("Product: ABC-12345-XYZ",), ("Item: DEF-67890-UVW",)]
df = spark.createDataFrame(data, ["description"])

# Extract different parts using idx 0 (full match), 1 (first group), 2 (second group)
df_multi = df.withColumn("full_code", regexp_extract("description", r"([A-Z]{3})-(\d{5})-([A-Z]{3})", 0)) \
             .withColumn("prefix", regexp_extract("description", r"([A-Z]{3})-(\d{5})-([A-Z]{3})", 1)) \
             .withColumn("number", regexp_extract("description", r"([A-Z]{3})-(\d{5})-([A-Z]{3})", 2)) \
             .withColumn("suffix", regexp_extract("description", r"([A-Z]{3})-(\d{5})-([A-Z]{3})", 3))

df_multi.show(truncate=False)

+----------------------+-------------+------+------+------+
|description           |full_code    |prefix|number|suffix|
+----------------------+-------------+------+------+------+
|Product: ABC-12345-XYZ|ABC-12345-XYZ|ABC   |12345 |XYZ   |
|Item: DEF-67890-UVW   |DEF-67890-UVW|DEF   |67890 |UVW   |
+----------------------+-------------+------+------+------+



## 2. `regexp_replace()`

Replaces all substrings matching a regex pattern with a replacement string.

**Syntax:** `regexp_replace(str, pattern, replacement)`
- `str`: Column to perform replacement on
- `pattern`: Regular expression pattern to match
- `replacement`: String to replace matches with

In [55]:
from pyspark.sql.functions import regexp_replace

data = [("Phone: 123-456-7890",), ("Call: 987-654-3210",)]
df = spark.createDataFrame(data, ["text"])

# Mask phone numbers
df_masked = df.withColumn("masked", regexp_replace("text", r"\d{3}-\d{3}-\d{4}", "XXX-XXX-XXXX"))
df_masked.show(truncate=False)

+-------------------+-------------------+
|text               |masked             |
+-------------------+-------------------+
|Phone: 123-456-7890|Phone: XXX-XXX-XXXX|
|Call: 987-654-3210 |Call: XXX-XXX-XXXX |
+-------------------+-------------------+



## 3. `rlike()`

Returns a boolean column indicating whether the string matches the regex pattern.

**Syntax:** `column.rlike(pattern)` or `rlike(column, pattern)`

In [56]:
from pyspark.sql.functions import col

data = [("apple",), ("banana",), ("apricot",), ("grape",)]
df = spark.createDataFrame(data, ["fruit"])

# Filter fruits starting with 'a'
df_filtered = df.filter(col("fruit").rlike("^a"))
df_filtered.show()

+-------+
|  fruit|
+-------+
|  apple|
|apricot|
+-------+

